In [0]:
%run "../common/define_path"

In [0]:
tracking_raw_path
tracking_br_path

In [0]:
def read_ordertracking_data():
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
    from pyspark.sql.functions import current_timestamp
  
    ordertracking_schema = StructType([
    StructField("trackingid",StringType()),
    StructField("shipmentno",StringType()),
    StructField("orderstatus",StringType()),
    StructField("createdon",StringType())
    ])

    rawordertracking_stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option('cloudFiles.schemaLocation',f'{tracking_br_path}/schema')
        .option('header','true')
        .schema(ordertracking_schema)
        .load(tracking_raw_path+'/ordertracking/')
        .withColumn("ingestion_time",current_timestamp())
    )
    return rawordertracking_stream

In [0]:
def write_ordertracking_Data(StreamingDF):
    write_Stream = (
        StreamingDF.writeStream
        .format("delta")
        .option("checkpointLocation", f"{tracking_br_path}/checkpoint/ordertracking/")
        .trigger(availableNow=True)
        .toTable("dev.tracking_br.ordertracking")
    )
    write_Stream.awaitTermination()

In [0]:
df_ordertracking = read_ordertracking_data()
write_ordertracking_Data(df_ordertracking)